# 簡單的RNN實作

## 程式參考來源：
- https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html
- https://pytorch.org/docs/stable/generated/nn.RNN.html#nn.RNN
- https://pytorch.org/text/stable/vocab.html
- https://pytorch.org/text/stable/functional.html#to-tensor
- https://pytorch.org/tutorials/beginner/text_sentiment_ngrams_tutorial.html


## 載入相關套件

In [1]:
import string
from collections import Counter, OrderedDict
from typing import List, Tuple

import numpy as np
import torch
from torch import nn, optim
from text_utils import GloVe, Vocab, get_tokenizer, to_tensor, truncate, vocab

## 嵌入層測試

In [2]:
x = torch.LongTensor([[0, 1, 2], [3, 4, 5]])
embeds = nn.Embedding(6, 5)
print(embeds(x))

tensor([[[-0.2916, -0.6667,  0.5122,  1.1526, -0.5012],
         [-1.9055, -0.7723, -1.7388, -0.4932,  0.3545],
         [ 1.4051, -1.3785, -0.6905,  0.1731, -0.1962]],

        [[-0.3244, -0.6321, -0.8768, -0.3236, -0.2689],
         [-0.4994, -0.4638, -0.0822, -0.6439,  0.3504],
         [-0.3657,  0.0861,  0.7475, -0.3471,  1.0824]]],
       grad_fn=<EmbeddingBackward0>)


In [3]:
embeds.weight

Parameter containing:
tensor([[-0.2916, -0.6667,  0.5122,  1.1526, -0.5012],
        [-1.9055, -0.7723, -1.7388, -0.4932,  0.3545],
        [ 1.4051, -1.3785, -0.6905,  0.1731, -0.1962],
        [-0.3244, -0.6321, -0.8768, -0.3236, -0.2689],
        [-0.4994, -0.4638, -0.0822, -0.6439,  0.3504],
        [-0.3657,  0.0861,  0.7475, -0.3471,  1.0824]], requires_grad=True)

In [4]:
x = torch.LongTensor([[1, 2, 3], [4, 5, 6]])
embeds = nn.Embedding(7, 5)
print(embeds(x))

tensor([[[-1.5574, -0.2986,  0.6524, -0.9540,  0.4357],
         [ 0.6443,  0.3168, -1.7078,  0.3461, -0.7417],
         [ 0.9882, -0.8443,  0.2247,  1.2574, -0.5207]],

        [[-1.1651,  0.7352, -2.3098, -0.1542,  1.4660],
         [-0.2264, -0.2765,  0.7830, -1.6544, -2.2910],
         [ 1.6857, -0.6730, -1.4594,  1.4304,  0.4307]]],
       grad_fn=<EmbeddingBackward0>)


In [5]:
embeds = nn.Embedding(6, 5)
x1 = torch.LongTensor([[0, 1, 2]])
x2 = torch.LongTensor([[3, 4]])
print(embeds(x1))
print(embeds(x2))
embeds.weight

tensor([[[-1.8707, -0.8831, -1.4209,  0.3838, -0.7183],
         [-0.9845, -0.7614, -0.2024, -1.8817,  0.4492],
         [-0.9507, -0.6147, -1.0422, -0.7121,  0.1006]]],
       grad_fn=<EmbeddingBackward0>)
tensor([[[ 0.6120,  1.2733,  0.0884, -0.8898,  0.6336],
         [ 1.2068,  2.1475,  0.1079,  1.3017,  0.2622]]],
       grad_fn=<EmbeddingBackward0>)


Parameter containing:
tensor([[-1.8707, -0.8831, -1.4209,  0.3838, -0.7183],
        [-0.9845, -0.7614, -0.2024, -1.8817,  0.4492],
        [-0.9507, -0.6147, -1.0422, -0.7121,  0.1006],
        [ 0.6120,  1.2733,  0.0884, -0.8898,  0.6336],
        [ 1.2068,  2.1475,  0.1079,  1.3017,  0.2622],
        [ 0.0110,  0.4365,  0.5549,  0.2106, -1.3764]], requires_grad=True)

In [6]:
embeds = nn.Embedding(6, 5, 5)
x1 = torch.LongTensor([[0, 1, 2]])
x2 = torch.LongTensor([[3, 4]])
x3 = torch.LongTensor([[3, 4]])
print(embeds(x1))
print(embeds(x2))
print(embeds(x3))
embeds.weight

tensor([[[-1.2799, -0.8275, -2.2138,  0.2604,  1.3617],
         [ 1.4198, -0.3455, -0.8153,  0.1793, -0.5748],
         [ 0.1824, -0.1263,  0.9018,  0.1076, -1.2142]]],
       grad_fn=<EmbeddingBackward0>)
tensor([[[-1.5012,  1.2367,  0.7578,  0.8577,  1.2578],
         [-0.5357, -0.9482,  0.2924, -0.8291, -0.6816]]],
       grad_fn=<EmbeddingBackward0>)
tensor([[[-1.5012,  1.2367,  0.7578,  0.8577,  1.2578],
         [-0.5357, -0.9482,  0.2924, -0.8291, -0.6816]]],
       grad_fn=<EmbeddingBackward0>)


Parameter containing:
tensor([[-1.2799, -0.8275, -2.2138,  0.2604,  1.3617],
        [ 1.4198, -0.3455, -0.8153,  0.1793, -0.5748],
        [ 0.1824, -0.1263,  0.9018,  0.1076, -1.2142],
        [-1.5012,  1.2367,  0.7578,  0.8577,  1.2578],
        [-0.5357, -0.9482,  0.2924, -0.8291, -0.6816],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000]], requires_grad=True)

In [7]:
# 測試資料
word_to_ix = {"hello": 0, "world": 1}
# 詞彙表(vocabulary)含2個單字, 轉換為5維的向量
embeds = nn.Embedding(2, 5)
# 測試 hello
lookup_tensor = torch.LongTensor([word_to_ix["hello"]])
hello_embed = embeds(lookup_tensor)
print(hello_embed)

tensor([[-0.4328, -0.3991,  0.3527,  2.0459,  1.0456]],
       grad_fn=<EmbeddingBackward0>)


## RNN層測試

In [8]:
torch.randn(5, 3, 10).shape

torch.Size([5, 3, 10])

In [9]:
# 測試資料
input = torch.randn(5, 10)
# 建立 RNN 物件
rnn = nn.RNN(10, 20, 2)
# RNN 處理
output: torch.Tensor
hn: torch.Tensor
output, hn = rnn(input)
# 顯示輸出及隱藏層的維度
print(output.shape, hn.shape)

torch.Size([5, 20]) torch.Size([2, 20])


In [10]:
# 測試資料
input = torch.randn(5, 4, 10)
# 建立 RNN 物件
rnn = nn.RNN(10, 20, 2)
# RNN 處理
output: torch.Tensor
hn: torch.Tensor
output, hn = rnn(input)
# 顯示輸出及隱藏層的維度
print(output.shape, hn.shape)

torch.Size([5, 4, 20]) torch.Size([2, 4, 20])


In [11]:
# 測試資料
input = torch.randn(5, 3, 10)
# 建立 RNN 物件
rnn = nn.RNN(10, 20, 2)
# 隱藏層的輸入
h0 = torch.randn(2, 3, 20)
# RNN 處理
output: torch.Tensor
hn: torch.Tensor
output, hn = rnn(input, h0)
# 顯示輸出及隱藏層的維度
print(output.shape, hn.shape)

torch.Size([5, 3, 20]) torch.Size([2, 3, 20])


## 分詞

In [12]:
tokenizer = get_tokenizer('basic_english')

text = 'Could have done better.'
tokenizer(text)

['could', 'have', 'done', 'better', '.']

## 詞彙表處理

In [13]:
# BOW 統計
counter = Counter(tokenizer(text))
# 依出現次數降冪排列
sorted_by_freq_tuples = sorted(counter.items(), key=lambda x: x[1], reverse=True)
# 建立詞彙字典
ordered_dict = OrderedDict(sorted_by_freq_tuples)

# 建立詞彙表物件，並加一個未知單字(unknown)的索引值
vocab_object = vocab(ordered_dict, specials=["<unk>"])
# 設定詞彙表預設值為未知單字(unknown)的索引值
vocab_object.set_default_index(vocab_object["<unk>"])

# 測試
vocab_object['done']

3

In [14]:
vocab_object.get_itos()

['<unk>', 'could', 'have', 'done', 'better', '.']

In [15]:
vocab_object.__len__()

6

In [16]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
def create_vocabulary(text_list: List[str]) -> Tuple[Vocab, List[str], List[List[int]]]:
    # 取得標點符號
    stopwords = list(string.punctuation)

    # 去除標點符號
    clean_text_list: List[str] = []
    clean_tokens_list: List[str] = []
    for text in text_list:
        tokens = tokenizer(text)
        clean_tokens: List[str] = []
        for w in tokens:
            if w not in stopwords:
                clean_tokens.append(w)
        clean_tokens_list += clean_tokens
        clean_text_list.append(' '.join(clean_tokens))

    # 建立詞彙表物件
    counter = Counter(clean_tokens_list)
    sorted_by_freq_tuples = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    ordered_dict = OrderedDict(sorted_by_freq_tuples)
    vocab_object = vocab(ordered_dict, specials=["<unk>"])
    vocab_object.set_default_index(vocab_object["<unk>"])

    # 將輸入字串轉為索引值：自詞彙表物件查詢索引值
    clean_index_list: List[List[int]] = []
    for clean_text in clean_text_list:
        clean_index_list.append(vocab_object.lookup_indices(clean_text.split(' ')))

    # 輸出 詞彙表物件、去除標點符號的字串陣列、字串陣列的索引值
    return vocab_object, clean_text_list, clean_index_list

## 測試

In [18]:
docs = [
    'Well done!',
    'Good work',
    'Great effort',
    'nice work',
    'Excellent!',
    'Weak',
    'Poor effort!',
    'not good',
    'poor work',
    'Could have done better.',
]

vocab_object, clean_text_list, clean_index_list = create_vocabulary(docs)
vocab_object.get_itos()

['<unk>',
 'work',
 'done',
 'good',
 'effort',
 'poor',
 'well',
 'great',
 'nice',
 'excellent',
 'weak',
 'not',
 'could',
 'have',
 'better']

In [19]:
clean_text_list

['well done',
 'good work',
 'great effort',
 'nice work',
 'excellent',
 'weak',
 'poor effort',
 'not good',
 'poor work',
 'could have done better']

In [20]:
clean_index_list

[[6, 2],
 [3, 1],
 [7, 4],
 [8, 1],
 [9],
 [10],
 [5, 4],
 [11, 3],
 [5, 1],
 [12, 13, 2, 14]]

# 整合以上功能，實作一個簡單的案例，說明相關的處理程序

## 建立詞彙表：整理輸入語句，截長補短，使語句長度一致。

In [21]:
maxlen = 4  # 語句最大字數
# 測試資料
docs = [
    'Well done!',
    'Good work',
    'Great effort',
    'nice work',
    'Excellent!',
    'Weak',
    'Poor effort!',
    'not good',
    'poor work',
    'Could have done better',
]

vocab_object, clean_text_list, clean_index_list = create_vocabulary(docs)

# 若字串過長，刪除多餘單字
clean_index_list = truncate(clean_index_list, maxlen)

# 若字串長度不足，後面補 0
while len(clean_index_list[0]) < maxlen:
    clean_index_list[0] += [0]
to_tensor(clean_index_list, 0)  # 0:不足補0

tensor([[ 6,  2,  0,  0],
        [ 3,  1,  0,  0],
        [ 7,  4,  0,  0],
        [ 8,  1,  0,  0],
        [ 9,  0,  0,  0],
        [10,  0,  0,  0],
        [ 5,  4,  0,  0],
        [11,  3,  0,  0],
        [ 5,  1,  0,  0],
        [12, 13,  2, 14]])

In [22]:
# 測試
embeds = nn.Embedding(vocab_object.__len__(), 5)
X = to_tensor(clean_index_list, 0)  # 0:不足補0
embed_output = embeds(X)
print(embed_output.shape)

torch.Size([10, 4, 5])


## 加上完全連接層(Linear)

In [ ]:
class RecurrentNet(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_class: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim * maxlen, num_class)  # 要乘以 maxlen
        self.embed_dim = embed_dim
        self.init_weights()

    def init_weights(self) -> None:
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text: torch.Tensor) -> torch.Tensor:
        embedded: torch.Tensor = self.embedding(text)
        out: torch.Tensor = embedded.reshape(embedded.size(0), -1)  # 轉換成1維
        return self.fc(out)


model = RecurrentNet(vocab_object.__len__(), 10, 1)

## 另一種寫法，使用EmbeddingBag

In [ ]:
class RecurrentNet(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_class: int) -> None:
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, num_class)
        self.embed_dim = embed_dim
        self.init_weights()

    def init_weights(self) -> None:
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text: torch.Tensor) -> torch.Tensor:
        embedded: torch.Tensor = self.embedding(text)
        return self.fc(embedded)


model = RecurrentNet(vocab_object.__len__(), 10, 1)

In [ ]:
# 定義 10 個語句的正面(1)或負面(0)的情緒
y = torch.FloatTensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
X = to_tensor(clean_index_list, 0)  # 0:不足補0

# 指定優化器、損失函數
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

# 模型訓練
for epoch in range(1000):
    outputs: torch.Tensor = model(X)
    optimizer.zero_grad()
    loss: torch.Tensor = criterion(outputs.reshape(-1), y)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        # print(outputs.shape)
        print(f"Epoch: {epoch}, loss: {loss.item():1.5f}")

Epoch: 0, loss: 0.69591
Epoch: 100, loss: 0.33837
Epoch: 200, loss: 0.17781
Epoch: 300, loss: 0.09236
Epoch: 400, loss: 0.03795
Epoch: 500, loss: 0.01102
Epoch: 600, loss: 0.00265
Epoch: 700, loss: 0.00080
Epoch: 800, loss: 0.00038
Epoch: 900, loss: 0.00023


In [26]:
# 模型評估
model.eval()
model(X)

tensor([[ 1.0006e+00],
        [ 9.9200e-01],
        [ 1.0141e+00],
        [ 9.9074e-01],
        [ 9.9759e-01],
        [-1.0333e-03],
        [-1.9752e-02],
        [ 3.5730e-03],
        [ 2.4688e-02],
        [-1.8001e-05]], grad_fn=<AddmmBackward0>)

In [ ]:
# 測試資料
test_docs = ['great effort', 'well done', 'poor effort']

# 轉成數值
clean_index_list: List[List[int]] = []
for text in test_docs:
    clean_index_list.append(vocab_object.lookup_indices(text.split(' ')))
while len(clean_index_list[0]) < maxlen:
    clean_index_list[0] += [0]

clean_index_list = truncate(clean_index_list, maxlen)
X = to_tensor(clean_index_list, 0)  # 0:不足補0
model(X)

tensor([[ 1.0141],
        [ 1.0006],
        [-0.0198]], grad_fn=<AddmmBackward0>)

## 使用詞向量(Word2Vec)

## 讀取 GloVe 50維的詞向量，轉換為GloVe 50維的詞向量

In [28]:
# https://pytorch.org/text/stable/vocab.html#glove
examples = ['great']
vec = GloVe(name='6B', dim=50)
ret = vec.get_vecs_by_tokens(examples, lower_case_backup=True)
ret

tensor([[-0.0266,  1.3357, -1.0280, -0.3729,  0.5201, -0.1270, -0.3543,  0.3782,
         -0.2972,  0.0939, -0.0341,  0.9296, -0.1402, -0.6330,  0.0208, -0.2153,
          0.9692,  0.4765, -1.0039, -0.2401, -0.3632, -0.0048, -0.5148, -0.4626,
          1.2447, -1.8316, -1.5581, -0.3747,  0.5336,  0.2088,  3.2209,  0.6455,
          0.3744, -0.1766, -0.0242,  0.3379, -0.4190,  0.4008, -0.1145,  0.0512,
         -0.1521,  0.2986, -0.4405,  0.1109, -0.2463,  0.6625, -0.2695, -0.4966,
         -0.4162, -0.2549]])

In [29]:
vec.vectors.size()

torch.Size([400000, 50])

In [30]:
vec.stoi['great']

353

## Embedding 不需訓練，直接設定嵌入層權重

In [ ]:
class RecurrentNet(nn.Module):
    def __init__(self, weights_matrix: torch.Tensor, num_embeddings: int, embedding_dim: int, num_class: int) -> None:
        super().__init__()
        self.embedding = nn.EmbeddingBag(num_embeddings, embedding_dim)
        # 設定嵌入層權重
        self.embedding.load_state_dict({'weight': weights_matrix})
        self.fc = nn.Linear(embedding_dim, num_class)

    def forward(self, text: torch.Tensor) -> torch.Tensor:
        embedded: torch.Tensor = self.embedding(text)
        return self.fc(embedded)

## 測試資料轉換

In [ ]:
docs = [
    'Well done!',
    'Good work',
    'Great effort',
    'nice work',
    'Excellent!',
    'Weak',
    'Poor effort!',
    'not good',
    'poor work',
    'Could have done better',
]

# 將詞彙表轉為詞向量
stopwords = list(string.punctuation)
clean_text_list: List[List[str]] = []
clean_tokens_list: List[str] = []
for i, text in enumerate(docs):
    tokens = tokenizer(text.lower())
    clean_tokens: List[str] = []
    for w in tokens:
        if w not in stopwords:
            clean_tokens.append(w)
    clean_tokens_list += clean_tokens
    clean_text_list.append(clean_tokens)
    tokens_vec = vec.get_vecs_by_tokens(clean_tokens)
vocab_list = list(set(clean_tokens_list))
weights_matrix = vec.get_vecs_by_tokens(vocab_list)

In [33]:
# 定義 10 個語句的正面(1)或負面(0)的情緒
y = torch.FloatTensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
X = torch.LongTensor(np.zeros((len(docs), maxlen)))
for i, item in enumerate(clean_text_list):
    for j, token in enumerate(item):
        if token in vocab_list:
            X[i, j] = vocab_list.index(token)
X

tensor([[ 7,  3,  0,  0],
        [10,  9,  0,  0],
        [ 6,  2,  0,  0],
        [ 0,  9,  0,  0],
        [13,  0,  0,  0],
        [ 1,  0,  0,  0],
        [ 8,  2,  0,  0],
        [ 5, 10,  0,  0],
        [ 8,  9,  0,  0],
        [11, 12,  3,  4]])

In [34]:
vocab_list

['nice',
 'weak',
 'effort',
 'done',
 'better',
 'not',
 'great',
 'well',
 'poor',
 'work',
 'good',
 'could',
 'have',
 'excellent']

In [ ]:
# 建立模型物件
model = RecurrentNet(torch.FloatTensor(weights_matrix), len(vocab_list), 50, 1)

# 指定優化器、損失函數
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

# 模型訓練
for epoch in range(1000):
    outputs: torch.Tensor = model(X)
    optimizer.zero_grad()
    loss: torch.Tensor = criterion(outputs.reshape(-1), y)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        # print(outputs.shape)
        print(f"Epoch: {epoch}, loss: {loss.item():1.5f}")

Epoch: 0, loss: 1.04764
Epoch: 100, loss: 0.13467
Epoch: 200, loss: 0.05173
Epoch: 300, loss: 0.00904
Epoch: 400, loss: 0.00185
Epoch: 500, loss: 0.00073
Epoch: 600, loss: 0.00029
Epoch: 700, loss: 0.00010
Epoch: 800, loss: 0.00004
Epoch: 900, loss: 0.00001


In [36]:
# 模型評估
model.eval()
model(X)

tensor([[ 1.0001e+00],
        [ 9.9904e-01],
        [ 1.0023e+00],
        [ 9.9674e-01],
        [ 1.0005e+00],
        [ 2.2911e-04],
        [-3.1535e-03],
        [ 6.4342e-04],
        [ 4.2911e-03],
        [-5.1029e-05]], grad_fn=<AddmmBackward0>)

In [ ]:
# 測試資料
test_docs = ['great effort', 'well done', 'poor effort']

# 轉成數值
X = torch.LongTensor(np.zeros((len(test_docs), maxlen)))
clean_text_list: List[List[str]] = []
for i, text in enumerate(test_docs):
    tokens = tokenizer(text.lower())
    clean_tokens: List[str] = []
    for w in tokens:
        if w not in stopwords:
            clean_tokens.append(w)
    clean_text_list.append(clean_tokens)

for i, item in enumerate(clean_text_list):
    for j, token in enumerate(item):
        if token in vocab_list:
            X[i, j] = vocab_list.index(token)

# 預測
model.eval()
model(X)

tensor([[ 1.0023],
        [ 1.0001],
        [-0.0032]], grad_fn=<AddmmBackward0>)

## 將整個詞向量設定為嵌入層權重

In [ ]:
class RecurrentNet2(nn.Module):
    def __init__(self, vec: torch.Tensor, embedding_dim: int, num_class: int) -> None:
        super().__init__()
        # 將整個詞向量設定為嵌入層權重，且嵌入層設為不訓練
        self.embedding = nn.EmbeddingBag.from_pretrained(vec, freeze=True)
        self.fc = nn.Linear(embedding_dim, num_class)

    def forward(self, text: torch.Tensor) -> torch.Tensor:
        embedded: torch.Tensor = self.embedding(text)
        return self.fc(embedded)


model = RecurrentNet2(vec.vectors, vec.dim, 1)

In [39]:
# 測試資料
docs = [
    'Well done!',
    'Good work',
    'Great effort',
    'nice work',
    'Excellent!',
    'Weak',
    'Poor effort!',
    'not good',
    'poor work',
    'Could have done better',
]

# 轉成數值
X = torch.LongTensor(np.zeros((len(docs), maxlen)))

for i, text in enumerate(docs):
    tokens = tokenizer(text.lower())
    clean_tokens = []
    j = 0
    for w in tokens:
        if w not in stopwords:
            # 轉成詞向量索引值
            X[i, j] = vec.stoi[w]
            j += 1
X

tensor([[ 143,  751,    0,    0],
        [ 219,  161,    0,    0],
        [ 353,  968,    0,    0],
        [3082,  161,    0,    0],
        [4345,    0,    0,    0],
        [2690,    0,    0,    0],
        [ 992,  968,    0,    0],
        [  36,  219,    0,    0],
        [ 992,  161,    0,    0],
        [  94,   33,  751,  439]])

In [ ]:
# 指定優化器、損失函數
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

# 模型訓練
for epoch in range(1000):
    outputs: torch.Tensor = model(X)
    optimizer.zero_grad()
    loss: torch.Tensor = criterion(outputs.reshape(-1), y)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        # print(outputs.shape)
        print(f"Epoch: {epoch}, loss: {loss.item():1.5f}")

model.eval()
model(X)

Epoch: 0, loss: 0.24673
Epoch: 100, loss: 0.09346
Epoch: 200, loss: 0.04481
Epoch: 300, loss: 0.02528
Epoch: 400, loss: 0.01517
Epoch: 500, loss: 0.00931
Epoch: 600, loss: 0.00587
Epoch: 700, loss: 0.00385
Epoch: 800, loss: 0.00262
Epoch: 900, loss: 0.00184


tensor([[ 0.9770],
        [ 0.9270],
        [ 1.0260],
        [ 1.0083],
        [ 1.0090],
        [-0.0115],
        [-0.0339],
        [ 0.0384],
        [ 0.0595],
        [ 0.0011]], grad_fn=<AddmmBackward0>)

In [41]:
# 測試資料
test_docs = ['great job', 'well done', 'poor job']

# 轉成數值
X = torch.LongTensor(np.zeros((len(test_docs), maxlen)))
for i, text in enumerate(test_docs):
    tokens = tokenizer(text.lower())
    clean_tokens = []
    j = 0
    for w in tokens:
        if w not in stopwords:
            X[i, j] = vec.stoi[w]
            j += 1
X

tensor([[353, 664,   0,   0],
        [143, 751,   0,   0],
        [992, 664,   0,   0]])

In [42]:
# 預測
model.eval()
model(X)

tensor([[ 0.5607],
        [ 0.9770],
        [-0.4992]], grad_fn=<AddmmBackward0>)